# IndiVoice-DeepASR: Extended Training (2000 -> 4000 Steps)

This notebook resumes training from the Step 2000 checkpoint and extends it to Step 4000 for maximum accuracy.

In [ ]:
import os, shutil

# 1. Configuration
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'
MODEL_REPO = "purvansh01/whisper-indian-lora"
OUTPUT_DIR = "/kaggle/working/models/whisper-indian-lora"
REPO_DIR = "/kaggle/working/IndiVoice-DeepASR"

print("[LOG] Environment and paths initialized.")

In [ ]:
# 2. Fast Repository Clone
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("[LOG] Cleaned old repo folder.")

print("[LOG] Starting Git Clone...")
!git clone --depth 1 https://github.com/purvanshjoshi/IndiVoice-DeepASR.git {REPO_DIR}
print("[LOG] Git Clone Finished.")

%cd {REPO_DIR}

In [ ]:
# 3. Install Dependencies (Classic Stability Mode)
print("[LOG] Cleaning old installations...")
!pip uninstall -y bitsandbytes peft transformers accelerate

print("[LOG] Installing Classic Stability Stack...")
!pip install -q -U torchao
!pip install -q transformers==4.44.2 peft==0.12.0 accelerate==0.34.2 bitsandbytes==0.41.1 datasets
print("[LOG] All dependencies installed.")

In [ ]:
# 4. Prepare Data & Download Checkpoint
import os
from huggingface_hub import snapshot_download

os.makedirs("data/processed", exist_ok=True)
manifest_path = "data/processed/svarah_manifest.json"
if not os.path.exists(manifest_path):
    print("[WARNING] Manifest not found. Running setup_kaggle.sh...")
    !chmod +x kaggle/setup_kaggle.sh
    !./kaggle/setup_kaggle.sh

print(f"[LOG] Downloading Step 2000 checkpoint...")
snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=OUTPUT_DIR,
    allow_patterns=["last-checkpoint/*", "adapter_model.safetensors", "adapter_config.json", "preprocessor_config.json"]
)

src_path = os.path.join(OUTPUT_DIR, "last-checkpoint")
dst_path = os.path.join(OUTPUT_DIR, "checkpoint-2000")

if os.path.exists(src_path):
    if os.path.exists(dst_path): shutil.rmtree(dst_path)
    os.rename(src_path, dst_path)
    print(f"[SUCCESS] Prepared checkpoint at {dst_path}")

In [ ]:
# 5. Launch Training
print("[LOG] Resuming Training to Step 4000...")
!accelerate launch src/train.py \
    --model_name "openai/whisper-medium" \
    --output_dir "/kaggle/working/models/whisper-indian-lora" \
    --train_manifest "data/processed/svarah_manifest.json" \
    --val_manifest "data/processed/svarah_manifest.json" \
    --batch_size 4 \
    --grad_accum 4 \
    --learning_rate 1e-4 \
    --max_steps 4000 \
    --hub_model_id "purvansh01/whisper-indian-lora"